# 🎬 CineForge · Colab 云端算力工作台（支持 H3 开源版 / LTX-2.5 / V2V 换角 / RelayVSR）

无需在本地配置任何 GPU 环境或安装模型权重，所有模型计算全部在 Google Colab 云端完成：
- **开箱支持多模型**：内置 **MiniMax-H3（开源 ComfyUI 4 步免 Box）**、**LTX-2.5**、**LTX-2.5 V2V 换角**、**RelayVSR 2× 超分**，并兼容已有 DramaPie Box 节点。
- **两种运行方式**：
  1. **连接本机 Web 工作台 (Farm)**：在本机运行 `bash studio/start.sh --with-farm` 打开前端界面，将终端显示的 `FARM_URL` 和 `FARM_TOKEN` 填入第一步，即可在本地 Web 选素材、写分镜、自动回传成片。
  2. **Colab 独立单镜试拍 (Standalone)**：不启动本机服务，直接在 Colab 内上传参考图/写提示词生成视频并下载。
- **推荐 GPU 配置**：Colab Pro/Pro+ 的 **G4 (96 GB) / H100 (80 GB) / A100 (80 GB)**。免费 T4 (15 GB) 显存不足以运行 22B/33B 视频模型。
- **防扣费 & 安全**：默认权重缓存在 Colab 本地高速盘 `/content`（避免撑爆 15 GB 免费 Drive），支持成片备份到 Google Drive、Colab 内直接上传/下载视频，以及闲置自动 `runtime.unassign()`。本笔记本不包含任何明文密钥或隧道地址。


In [ ]:
#@title 1️⃣ 参数配置（无需改代码，直接下拉选择即可）
RUN_MODE = "连接本机 Web 工作台 (Farm)"  #@param ["连接本机 Web 工作台 (Farm)", "Colab 独立单镜试拍 (Standalone)"]
MODEL_MODE = "MiniMax-H3 (开源4步免Box)"  #@param ["MiniMax-H3 (开源4步免Box)", "LTX-2.5 (原创图生/首帧)", "LTX-2.5 V2V (视频换角/变猫)", "RelayVSR X2 (2倍超分)", "MiniMax-H3 (自带Box加速版)"]
FARM_URL = "YOUR_FARM_URL"  #@param {type:"string"}
FARM_TOKEN = "YOUR_FARM_TOKEN"  #@param {type:"string"}
WORKER_NAME = "colab_gpu_01"  #@param {type:"string"}
IDLE_SHUTDOWN_MIN = 15  #@param {type:"slider", min:5, max:60, step:5}
USE_GDRIVE_CACHE = False  #@param {type:"boolean"}
BACKUP_OUTPUTS_TO_GDRIVE = True  #@param {type:"boolean"}
STANDALONE_PROMPT = "Photoreal cinematic sci-fi shot, warm tungsten key light, an orange tabby cat wearing a tiny space helmet looks out a spacecraft window at Earth, gentle ambient cabin hum and soft radio beep."  #@param {type:"string"}
STANDALONE_SECONDS = 5  #@param {type:"slider", min:4, max:8, step:1}

import os
ENGINE_MAP = {
    "MiniMax-H3 (开源4步免Box)": "h3-open",
    "LTX-2.5 (原创图生/首帧)": "ltx-2.5",
    "LTX-2.5 V2V (视频换角/变猫)": "ltx-2.5-v2v",
    "RelayVSR X2 (2倍超分)": "relayvsr",
    "MiniMax-H3 (自带Box加速版)": "minimax-h3-box",
}
ENGINE_KEY = ENGINE_MAP.get(MODEL_MODE, "h3-open")
IS_STANDALONE = RUN_MODE.startswith("Colab")
os.environ["CINEFORGE_ENGINE"] = ENGINE_KEY
os.environ["CINEFORGE_WORKER"] = WORKER_NAME
os.environ["CINEFORGE_IDLE_MIN"] = str(IDLE_SHUTDOWN_MIN)
print(f"[CineForge] 运行方式={RUN_MODE} | 模型={MODEL_MODE} ({ENGINE_KEY}) | 闲置自动释放={IDLE_SHUTDOWN_MIN}分钟")


In [ ]:
#@title 2️⃣ 显卡体检、读取 HF_TOKEN 并准备输出目录
import os, pathlib, shutil, subprocess

gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("[CineForge] 当前 GPU:", gpu_info or "未检测到 GPU（请在上方「代码执行程序 -> 更改运行时类型」中开启 GPU）")
disk_free_gb = shutil.disk_usage('/content').free / 1e9
print(f"[CineForge] /content 可用磁盘: {disk_free_gb:.1f} GB")
if "T4" in gpu_info:
    print("⚠️ 警告：T4 (15 GB) 显存不足以运行 22B/33B 视频模型，建议切换至 A100 / H100 / G4。")

# 优先从 Colab Secrets 读取 HF_TOKEN（无需手贴明文；联机模式也会自动从本机 .env 获取）
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        sec = userdata.get("HF_TOKEN")
        if sec:
            os.environ["HF_TOKEN"] = sec.strip()
            print("[CineForge] 已从 Colab Secrets 加载 HF_TOKEN")
    except Exception:
        pass

OUTPUT_DIR = pathlib.Path("/content/cineforge_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = pathlib.Path("/content/cineforge_cache")
GDRIVE_OUT = None
if USE_GDRIVE_CACHE or BACKUP_OUTPUTS_TO_GDRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        if BACKUP_OUTPUTS_TO_GDRIVE:
            GDRIVE_OUT = pathlib.Path("/content/drive/MyDrive/CineForgeOutputs")
            GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
            print("[CineForge] 成片将自动备份至 Google Drive:", GDRIVE_OUT)
        if USE_GDRIVE_CACHE:
            CACHE_ROOT = pathlib.Path("/content/drive/MyDrive/CineForgeCache/models")
            print("[CineForge] 已启用 Google Drive 权重缓存（请确认 Drive 剩余空间 >= 80 GB）:", CACHE_ROOT)
    except Exception as exc:
        print("[CineForge] 非 Colab 环境或未授权 Drive，使用本地临时目录:", exc)

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_ROOT / "hf")
os.environ["CINEFORGE_MODEL_CACHE"] = str(CACHE_ROOT)
print("[CineForge] 缓存目录就绪:", CACHE_ROOT, "| 成片目录:", OUTPUT_DIR)


In [ ]:
#@title 3️⃣ 启动云端模型计算（自动对接本机 Web 界面，或独立单镜出片）
import os, pathlib, shutil, urllib.parse, urllib.request

def release_colab_runtime(reason: str) -> None:
    print(f"[CineForge IdleGuard] {reason} -> 正在释放 Colab 显卡以节省 CU...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print("[CineForge IdleGuard] 非 Colab 环境跳过 unassign():", exc)

def sync_outputs_to_drive() -> None:
    if not BACKUP_OUTPUTS_TO_GDRIVE or GDRIVE_OUT is None:
        return
    for mp4 in sorted(pathlib.Path("/content/cineforge_outputs").glob("*.mp4")):
        dst = GDRIVE_OUT / mp4.name
        if not dst.exists() or dst.stat().st_size != mp4.stat().st_size:
            shutil.copy2(mp4, dst)
    print("[CineForge] 已同步成片至 Google Drive:", GDRIVE_OUT)

RAW_GITHUB = "https://raw.githubusercontent.com/junwenxiong/cineforge/main"
if IS_STANDALONE:
    print("[CineForge] 独立试拍模式：直接从 GitHub 拉取开源 MiniMax-H3 运行器并在 Colab 生成视频...")
    script_url = f"{RAW_GITHUB}/tools/film/lab/h3_comfy.py"
    ref_dir = pathlib.Path('/content/cineforge_uploads')
    ref_files = sorted(str(p) for p in ref_dir.glob('*') if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp')) if ref_dir.is_dir() else []
    worker_ns = {
        "B": "", "T": "", "W": WORKER_NAME,
        "STANDALONE_PROMPT": STANDALONE_PROMPT,
        "STANDALONE_REFS": ref_files,
        "STANDALONE_SECONDS": STANDALONE_SECONDS,
        "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
    }
    with urllib.request.urlopen(script_url, timeout=30) as resp:
        code = resp.read().decode("utf-8")
    exec(compile(code, script_url, "exec"), worker_ns)
    sync_outputs_to_drive()
elif not FARM_URL or FARM_URL == "YOUR_FARM_URL" or not FARM_TOKEN or FARM_TOKEN == "YOUR_FARM_TOKEN":
    print("[CineForge] 请先在本机运行 `bash studio/start.sh --with-farm`，然后把终端显示的 FARM_URL 与 FARM_TOKEN 填入第 1 步，再重新运行本单元。")
else:
    base = FARM_URL.rstrip("/")
    q = urllib.parse.urlencode({"t": FARM_TOKEN, "w": WORKER_NAME})
    if ENGINE_KEY == "ltx-2.5-v2v":
        script_url = f"{base}/lab/file/ltx_v2v.py?{q}"
        worker_ns = {"B": base, "T": FARM_TOKEN, "W": WORKER_NAME, "LAB_MODEL": "v2v_studio", "POLL_STUDIO": True, "IDLE_SHUTDOWN_MIN": IDLE_SHUTDOWN_MIN, "AUTO_UNASSIGN": True}
    elif ENGINE_KEY == "ltx-2.5":
        script_url = f"{base}/lab/file/ltx25.py?{q}"
        worker_ns = {"B": base, "T": FARM_TOKEN, "W": WORKER_NAME, "LAB_MODEL": "ltx_studio", "POLL_STUDIO": True, "IDLE_SHUTDOWN_MIN": IDLE_SHUTDOWN_MIN, "AUTO_UNASSIGN": True}
    elif ENGINE_KEY == "relayvsr":
        script_url = f"{base}/lab/file/relayvsr_film.py?{q}"
        worker_ns = {"B": base, "T": FARM_TOKEN, "W": WORKER_NAME}
    elif ENGINE_KEY == "minimax-h3-box":
        script_url = f"{base}/boot.py?{q}"
        worker_ns = {"B": base, "T": FARM_TOKEN, "W": WORKER_NAME, "FILMS": "", "IDLE_SHUTDOWN_MIN": IDLE_SHUTDOWN_MIN, "AUTO_UNASSIGN": True}
    else:
        script_url = f"{base}/lab/file/h3_comfy.py?{q}"
        worker_ns = {"B": base, "T": FARM_TOKEN, "W": WORKER_NAME, "FILMS": "", "IDLE_SHUTDOWN_MIN": IDLE_SHUTDOWN_MIN, "AUTO_UNASSIGN": True}
    os.environ["CINEFORGE_AUTO_UNASSIGN"] = "1"
    print(f"[CineForge] 节点 {WORKER_NAME} 正在拉取 {ENGINE_KEY} 工作脚本并接入本机 Web 派片服务...")
    try:
        with urllib.request.urlopen(script_url, timeout=30) as resp:
            code = resp.read().decode("utf-8")
        exec(compile(code, script_url, "exec"), worker_ns)
    finally:
        sync_outputs_to_drive()
        release_colab_runtime("任务执行结束或队列已清空")


In [ ]:
#@title 4️⃣ Colab 视频上传与下载工具箱（可选：直接上传参考素材 / 在 Colab 预览并下载成片）
ACTION = "列出并预览已生成的视频"  #@param ["列出并预览已生成的视频", "一键打包下载全部生成视频 (.zip)", "上传参考图/原片视频到 Colab"]
import pathlib, zipfile
from IPython.display import HTML, Video, display

out_dir = pathlib.Path("/content/cineforge_outputs")
up_dir = pathlib.Path("/content/cineforge_uploads")
out_dir.mkdir(parents=True, exist_ok=True)
up_dir.mkdir(parents=True, exist_ok=True)

if ACTION == "上传参考图/原片视频到 Colab":
    from google.colab import files
    uploaded = files.upload()
    for fname, raw in uploaded.items():
        dst = up_dir / pathlib.Path(fname).name
        dst.write_bytes(raw)
        print("已上传:", dst, f"({len(raw)/1e6:.2f} MB)")
elif ACTION == "一键打包下载全部生成视频 (.zip)":
    mp4s = sorted(out_dir.glob('*.mp4'))
    if not mp4s:
        print("暂无可下载的视频，请先运行第 3 步生成。")
    else:
        from google.colab import files
        zpath = pathlib.Path("/content/cineforge_outputs.zip")
        with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
            for p in mp4s:
                zf.write(p, arcname=p.name)
        print(f"已打包 {len(mp4s)} 个视频 -> {zpath}")
        files.download(str(zpath))
else:
    mp4s = sorted(out_dir.glob('*.mp4'))
    if not mp4s:
        print("目录为空：", out_dir)
    for p in mp4s:
        print(f"🎬 {p.name} ({p.stat().st_size/1e6:.2f} MB)")
        display(Video(str(p), embed=True, width=640))
